# ssh

> SSH-over-WebSocket ProxyCommand client, session store, and `.ssh/config` updater for Rustygate, Solveit, and Colab

In [ ]:
#| default_exp ssh

`jupyasyncclient.ssh` bridges OpenSSH's `ProxyCommand` (`stdin`/`stdout`) to an SSH-over-WebSocket endpoint carrying raw OpenSSH bytes in binary frames. It works with Rustygate (`/api/ssh`), Solveit (`/rusty/api/ssh`), and Google Colab (`/colab/ssh`).

The module stores per-host connection details in `$XDG_CONFIG_HOME/jupyasyncclient/ssh.json`, updates `~/.ssh/config` idempotently, and provides the `jac-ssh` CLI with `--colab`, `--solveit`, and `--proxy` modes.

In [ ]:
#| export
import json, os, shlex, shutil, subprocess, sys, threading
from contextlib import suppress
from pathlib import Path
from urllib.parse import urlencode, urlsplit, urlunsplit
from fastcore.basics import AttrDict
from fastcore.script import call_parse
from fastcore.utils import atomic_save
from fastcore.xdg import xdg_config_home
from websockets.exceptions import ConnectionClosed, InvalidStatus, WebSocketException
from websockets.sync.client import connect as ws_connect
from jupyasyncclient.core import _join_url

In [ ]:
#| hide
import tempfile
from fastcore.test import test_eq
from websockets.http11 import Response as WsResponse
from websockets.datastructures import Headers as WsHeaders
from websockets.sync.server import serve as ws_serve

## WebSocket URLs and public keys

Colab, Solveit, and bare Rustygate gateways share the same binary-frame SSH wire protocol and differ only in their endpoint URL and auth headers. `colab_ws_url`, `solveit_ws_url`, and `gateway_ws_url` normalize each target to a `ws://` or `wss://` endpoint.

In [ ]:
#| export
COLAB_PUBKEY_HEADER = 'X-Colab-Ssh-Pubkey'
SSH_KEY_NAMES = ('id_ed25519', 'id_ecdsa')

def _ws_scheme(scheme:str)->str: return {'http': 'ws', 'https': 'wss'}.get(scheme, scheme)

def colab_ws_url(
    url:str, # Colab runtime base URL (e.g. `https://<host>`)
    token:str, # Colab runtime proxy token
    mode:str='proxy', # Colab SSH client mode (`proxy`, `interactive`, or `cmd`)
)->str:
    "Build the `wss://{host}/colab/ssh` URL for a Colab runtime."
    u = urlsplit(url)
    q = urlencode([('colab-runtime-proxy-token', token), ('colab-client-agent', 'colab-cli'), ('colab-client-agent', f'ssh-mode-{mode}')])
    return urlunsplit((_ws_scheme(u.scheme), u.netloc, '/colab/ssh', q, ''))

In [ ]:
test_eq(colab_ws_url('https://rt.colab.dev', 'tok'),
    'wss://rt.colab.dev/colab/ssh?colab-runtime-proxy-token=tok&colab-client-agent=colab-cli&colab-client-agent=ssh-mode-proxy')

In [ ]:
#| export
def solveit_ws_url(
    url:str, # Solveit instance URL (bare origin, `/rusty`, or full `/rusty/api/ssh`)
)->str:
    "Normalize a Solveit instance URL to its `/rusty/api/ssh` WebSocket endpoint."
    u = urlsplit(url)
    p = u.path.rstrip('/')
    if not p.endswith('/api/ssh'): p = f'{p}/api/ssh' if p.endswith('/rusty') else f'{p}/rusty/api/ssh'
    return urlunsplit((_ws_scheme(u.scheme), u.netloc, p, u.query, ''))

def gateway_ws_url(
    url:str, # Gateway URL (HTTP(S) base or WebSocket endpoint)
    path:str='/api/ssh', # Endpoint path appended when `url` is a base URL
    token:str=None, # Optional gateway `?token=` query parameter
)->str:
    "Normalize a Rustygate or generic gateway URL to an SSH WebSocket endpoint."
    u = urlsplit(url)
    if u.path.rstrip('/').endswith('/ssh'):
        params = [p for p in (u.query, urlencode({'token': token}) if token else '') if p]
        return urlunsplit((_ws_scheme(u.scheme), u.netloc, u.path, '&'.join(params), ''))
    return _join_url(url, path, ws=True, params={'token': token} if token else None)

In [ ]:
test_eq(solveit_ws_url('https://demo.solve.it.com'), 'wss://demo.solve.it.com/rusty/api/ssh')
test_eq(solveit_ws_url('https://demo.solve.it.com/rusty/'), 'wss://demo.solve.it.com/rusty/api/ssh')
test_eq(solveit_ws_url('wss://demo.solve.it.com/rusty/api/ssh'), 'wss://demo.solve.it.com/rusty/api/ssh')
test_eq(gateway_ws_url('http://127.0.0.1:8787', token='sec'), 'ws://127.0.0.1:8787/api/ssh?token=sec')
test_eq(gateway_ws_url('ws://127.0.0.1:8787/api/ssh'), 'ws://127.0.0.1:8787/api/ssh')

`resolve_pubkey` reads an explicit key's `.pub` file (deriving it via `ssh-keygen -y -f` when only the private key exists) or auto-selects `~/.ssh/id_ed25519.pub` then `~/.ssh/id_ecdsa.pub`. RSA keys are skipped during auto-selection because Colab rejects them.

In [ ]:
#| export
def default_identity(
    ssh_dir:str|Path='~/.ssh', # Directory to scan for non-RSA key pairs
)->str|None:
    "Return the first `id_ed25519` or `id_ecdsa` path whose private and `.pub` files both exist."
    d = Path(ssh_dir).expanduser()
    is_home = d == Path('~/.ssh').expanduser()
    for n in SSH_KEY_NAMES:
        if (d/n).exists() and (d/f'{n}.pub').exists(): return f'~/.ssh/{n}' if is_home else str(d/n)
    return None

def _pubkey_from_private(path:Path)->str:
    if not path.exists(): raise FileNotFoundError(f'SSH key not found: {path}')
    res = subprocess.run(['ssh-keygen', '-y', '-f', str(path)], check=True, capture_output=True, text=True)
    pub = res.stdout.strip()
    if not pub: raise RuntimeError(f'ssh-keygen produced no public key for {path}')
    return pub

def resolve_pubkey(
    identity:str|Path=None, # Explicit public or private key path, or `None` to scan `ssh_dir`
    ssh_dir:str|Path='~/.ssh', # Directory scanned when `identity` is omitted
)->str:
    "Return the OpenSSH public key text to send in the WebSocket upgrade header."
    if identity:
        p = Path(identity).expanduser()
        pub_p, priv_p = (p, p.with_suffix('')) if p.suffix == '.pub' else (p.with_suffix('.pub'), p)
        if pub_p.exists() and (txt := pub_p.read_text().strip()): return txt
        return _pubkey_from_private(priv_p if priv_p.exists() else p)
    d = Path(ssh_dir).expanduser()
    for n in SSH_KEY_NAMES:
        cand = d/f'{n}.pub'
        if cand.exists() and (txt := cand.read_text().strip()): return txt
    raise FileNotFoundError(f'No ed25519 or ecdsa public key found in {d}. Run `ssh-keygen -t ed25519` or pass --identity.')

In [ ]:
with tempfile.TemporaryDirectory(prefix='jac-ssh-key-') as td:
    kp = Path(td)/'id_ed25519'
    subprocess.run(['ssh-keygen', '-q', '-t', 'ed25519', '-N', '', '-f', str(kp)], check=True)
    pub = kp.with_suffix('.pub').read_text().strip()
    test_eq(default_identity(td), str(kp))
    test_eq(resolve_pubkey(ssh_dir=td), pub)
    test_eq(resolve_pubkey(kp), pub)
    kp.with_suffix('.pub').unlink()
    assert resolve_pubkey(kp).startswith('ssh-ed25519 ')

## Session store and `.ssh/config`

`save_ssh_session`, `get_ssh_session`, and `remove_ssh_session` manage per-host connection settings in `$XDG_CONFIG_HOME/jupyasyncclient/ssh.json` (mode `0600`) so `ProxyCommand` lines stay short and never embed secrets directly in `~/.ssh/config`.

In [ ]:
#| export
def ssh_store_path(store:str|Path=None)->Path:
    "Return the SSH session store JSON path."
    return Path(store).expanduser() if store else xdg_config_home()/'jupyasyncclient'/'ssh.json'

def load_ssh_sessions(store:str|Path=None)->dict:
    "Load all saved SSH session entries from `store`."
    p = ssh_store_path(store)
    return json.loads(p.read_text()) if p.exists() else {}

In [ ]:
#| export
def save_ssh_session(
    host:str, # SSH Host alias key
    entry:dict, # Connection fields (`url`, `headers`, `token_file`, `pubkey_header`, etc.)
    store:str|Path=None, # Optional session store path
)->AttrDict:
    "Atomically save `entry` under `host` in the SSH session store (mode `0600`)."
    p = ssh_store_path(store)
    data = load_ssh_sessions(p)
    data[host] = {k: v for k, v in entry.items() if v is not None}
    p.parent.mkdir(parents=True, exist_ok=True, mode=0o700)
    with atomic_save(p, mode='w') as f: json.dump(data, f, indent=2)
    os.chmod(p, 0o600)
    return AttrDict(data[host])

def get_ssh_session(host:str, store:str|Path=None)->AttrDict|None:
    "Return the saved SSH session for `host`, or `None` if absent."
    entry = load_ssh_sessions(store).get(host)
    return AttrDict(entry) if entry is not None else None

def remove_ssh_session(host:str, store:str|Path=None)->bool:
    "Remove `host` from the SSH session store; return `True` if it existed."
    p = ssh_store_path(store)
    data = load_ssh_sessions(p)
    if host not in data: return False
    del data[host]
    with atomic_save(p, mode='w') as f: json.dump(data, f, indent=2)
    os.chmod(p, 0o600)
    return True

`update_ssh_config` appends a `Host <host>` stanza when absent, or replaces it when `replace=True`. `setup_ssh` combines session persistence and config updates, returning `ssh <host>`.

In [ ]:
#| export
def _host_matches(line:str, host:str)->bool:
    toks = line.split('#', 1)[0].split()
    return len(toks) > 1 and toks[0].lower() == 'host' and host in toks[1:]

def _has_ssh_host(cfg:str, host:str)->bool: return any(_host_matches(ln, host) for ln in cfg.splitlines())

def _replace_ssh_host(cfg:str, host:str, block:str)->str:
    lines, out, i = cfg.splitlines(keepends=True), [], 0
    while i < len(lines):
        if _host_matches(lines[i], host):
            out.append(block)
            i += 1
            while i < len(lines):
                toks = lines[i].split('#', 1)[0].split()
                if toks and toks[0].lower() in ('host', 'match'): break
                i += 1
            if i < len(lines) and not block.endswith('\n\n'): out.append('\n')
        else:
            out.append(lines[i])
            i += 1
    return ''.join(out)

In [ ]:
#| export
def _proxy_bin()->str:
    if p := shutil.which('jac-ssh'): return shlex.quote(p)
    return f'{shlex.quote(sys.executable)} -m jupyasyncclient.ssh'

def ssh_host_block(
    host:str, # SSH Host alias
    *,
    user:str=None, # Optional remote SSH username (`root` for Colab, `solveit` for Solveit)
    identity:str=None, # Optional `ed25519`/`ecdsa` key path (auto-detected if omitted)
    ssh_dir:Path=None, # Directory used for key auto-detection
    proxy_cmd:str=None, # Explicit `ProxyCommand` override
    store:str|Path=None, # Non-default session store path to pass to `jac-ssh --proxy`
)->str:
    "Render an OpenSSH `Host` stanza pointing `ProxyCommand` at `jac-ssh --proxy`."
    key = identity or default_identity(ssh_dir or Path('~/.ssh').expanduser())
    if not proxy_cmd:
        parts = [_proxy_bin(), '--proxy', '--host', shlex.quote(host)]
        if key: parts += ['--identity', shlex.quote(key)]
        if store: parts += ['--store', shlex.quote(str(Path(store).expanduser()))]
        proxy_cmd = ' '.join(parts)
    lines = [f'Host {host}']
    if user: lines.append(f'    User {user}')
    lines += ['    StrictHostKeyChecking no', '    UserKnownHostsFile /dev/null', '    LogLevel ERROR',
        '    ServerAliveInterval 30', f'    ProxyCommand {proxy_cmd}']
    if key: lines += [f'    IdentityFile {key}', '    IdentitiesOnly yes']
    return '\n'.join(lines) + '\n'

In [ ]:
#| export
def update_ssh_config(
    host:str, # SSH Host alias
    *,
    user:str=None, # Optional remote SSH username
    identity:str=None, # Optional `ed25519`/`ecdsa` key path
    ssh_config:str|Path='~/.ssh/config', # OpenSSH config file to update
    proxy_cmd:str=None, # Explicit `ProxyCommand` override
    store:str|Path=None, # Optional non-default session store path
    replace:bool=False, # Replace an existing `Host <host>` block instead of keeping it
)->Path:
    "Ensure a `Host <host>` entry exists in `ssh_config` (idempotent unless `replace=True`)."
    cfg_path = Path(ssh_config).expanduser()
    cfg = cfg_path.read_text() if cfg_path.exists() else ''
    block = ssh_host_block(host, user=user, identity=identity, ssh_dir=cfg_path.parent, proxy_cmd=proxy_cmd, store=store)
    if _has_ssh_host(cfg, host):
        if not replace: return cfg_path
        new_cfg = _replace_ssh_host(cfg, host, block)
    else:
        sep = '' if not cfg else ('\n\n' if not cfg.endswith('\n') else ('' if cfg.endswith('\n\n') else '\n'))
        new_cfg = cfg + sep + block
    cfg_path.parent.mkdir(parents=True, exist_ok=True, mode=0o700)
    if not cfg_path.exists(): cfg_path.touch(mode=0o600)
    cfg_path.write_text(new_cfg)
    return cfg_path

def setup_ssh(
    host:str, # SSH Host alias in `ssh_config` and key in the session store
    url:str=None, # Optional WebSocket or HTTP(S) target URL to save in the session store
    *,
    user:str=None, # Remote SSH username (`root` for Colab, `solveit` for Solveit)
    identity:str=None, # Optional `ed25519`/`ecdsa` key path (auto-detected if omitted)
    headers:dict=None, # Extra HTTP headers sent on the WebSocket upgrade
    token:str=None, # Optional gateway `?token=` query parameter
    bearer:str=None, # Optional `Authorization: Bearer <bearer>` token
    token_file:str|Path=None, # Optional file holding a Bearer token (read on each connect)
    pubkey_header:str=COLAB_PUBKEY_HEADER, # Upgrade header carrying the SSH public key
    ssh_config:str|Path='~/.ssh/config', # OpenSSH config file to update
    store:str|Path=None, # Optional session store path
    replace:bool=False, # Replace an existing `Host <host>` entry in `ssh_config`
    meta:dict=None, # Extra metadata to store alongside the session (e.g. `endpoint`, `expire_time`)
)->str:
    "Save `host` connection details (when given) and ensure `Host <host>` exists in `ssh_config`."
    if any(x is not None for x in (url, headers, token, bearer, token_file, meta)):
        entry = dict(meta or {})
        entry.update(url=url, user=user, identity=identity, headers=headers, token=token, bearer=bearer,
            token_file=str(Path(token_file).expanduser()) if token_file else None, pubkey_header=pubkey_header)
        save_ssh_session(host, entry, store=store)
    update_ssh_config(host, user=user, identity=identity, ssh_config=ssh_config, store=store, replace=replace)
    return f'ssh {host}'

In [ ]:
with tempfile.TemporaryDirectory(prefix='jac-ssh-cfg-') as td:
    cfg, st = Path(td)/'config', Path(td)/'ssh.json'
    test_eq(setup_ssh('solveit', solveit_ws_url('https://demo.solve.it.com'), user='solveit',
        token_file='~/.config/solveit/session', ssh_config=cfg, store=st), 'ssh solveit')
    first = cfg.read_text()
    assert 'Host solveit' in first and 'User solveit' in first and '--proxy --host solveit' in first
    setup_ssh('solveit', user='other', ssh_config=cfg, store=st)
    test_eq(cfg.read_text(), first)
    setup_ssh('solveit', user='other', ssh_config=cfg, store=st, replace=True)
    assert 'User other' in cfg.read_text() and 'User solveit' not in cfg.read_text()
    test_eq(get_ssh_session('solveit', store=st).url, 'wss://demo.solve.it.com/rusty/api/ssh')
    test_eq(remove_ssh_session('solveit', store=st), True)
    test_eq((remove_ssh_session('solveit', store=st), get_ssh_session('solveit', store=st)), (False, None))

## WebSocket-to-stdio bridge

`bridge_ws` copies raw OpenSSH bytes between `stdin`/`stdout` and binary WebSocket frames until either side closes. `proxy_ssh` loads a saved session (or direct CLI options), attaches the resolved SSH public key and auth headers, opens the WebSocket with compression disabled, and hands the connection to `bridge_ws`.

In [ ]:
#| export
def _explain_handshake_error(status:int, body:bytes)->str:
    snippet = body.decode('utf-8', errors='replace').strip()[:200]
    if status == 400 and 'missing pubkey' in snippet: return 'Server rejected request: missing SSH public key header.'
    if status == 400 and 'unsupported key type' in snippet: return 'Server rejected public key: unsupported key type (use ed25519 or ecdsa).'
    if status == 400: return f'Server rejected SSH request (HTTP 400): {snippet}'
    if status == 401: return 'Authentication failed (HTTP 401): token invalid or expired.'
    if status == 403: return f'Forbidden (HTTP 403): {snippet or "access denied"}'
    if status == 404: return 'Endpoint not found (HTTP 404): server does not expose this SSH WebSocket path.'
    if status == 429: return f'SSH session limit reached (HTTP 429): {snippet}'
    if status == 502: return 'Bad gateway (HTTP 502): upstream SSH server is unreachable.'
    return f'WebSocket upgrade rejected (HTTP {status}): {snippet}'

def bridge_ws(
    ws, # Connected `websockets.sync.client.ClientConnection`
    stdin_fd:int=None, # Input file descriptor (defaults to `sys.stdin.buffer.fileno()`)
    stdout=None, # Binary output stream (defaults to `sys.stdout.buffer`)
)->int:
    "Bridge raw bytes bidirectionally between `stdin_fd`/`stdout` and binary frames on `ws`."
    if stdin_fd is None: stdin_fd = sys.stdin.buffer.fileno()
    if stdout is None: stdout = sys.stdout.buffer
    def _stdin_to_ws():
        try:
            while data := os.read(stdin_fd, 65536): ws.send(data)
        except (OSError, WebSocketException): pass
        finally:
            with suppress(Exception): ws.close()
    threading.Thread(target=_stdin_to_ws, daemon=True).start()
    try:
        for frame in ws:
            if isinstance(frame, str): frame = frame.encode('utf-8')
            stdout.write(frame)
            stdout.flush()
    except (ConnectionClosed, WebSocketException, OSError): pass
    finally:
        with suppress(Exception): ws.close()
    return 0

In [ ]:
#| export
def _read_token_file(path:str|Path)->str:
    p = Path(path).expanduser()
    if not p.exists(): raise FileNotFoundError(f'Token file not found: {p} (run `solveit-login` first)')
    tok = p.read_text().strip()
    if not tok: raise RuntimeError(f'Token file is empty: {p}')
    return tok

def proxy_ssh(
    host:str=None, # Saved session alias to load from `store`
    *,
    url:str=None, # Direct WebSocket or HTTP(S) URL (overrides stored session URL)
    identity:str=None, # Path to public or private SSH key
    token:str=None, # Optional `?token=` query parameter
    bearer:str=None, # Optional `Authorization: Bearer` token
    token_file:str|Path=None, # Optional file holding a Bearer token
    headers:dict=None, # Extra HTTP upgrade headers
    pubkey_header:str=None, # Upgrade header name for the SSH public key
    store:str|Path=None, # Optional session store path
    stdin_fd:int=None, # Override input fd (for testing)
    stdout=None, # Override output stream (for testing)
)->int:
    "Connect to an SSH-over-WebSocket endpoint and bridge stdio for OpenSSH `ProxyCommand`."
    sess = get_ssh_session(host, store=store) if host else None
    if host and not sess and not url:
        print(f'[jac-ssh] session {host!r} not found in {ssh_store_path(store)}.', file=sys.stderr)
        return 2
    sess = sess or AttrDict()
    target = url or sess.get('url')
    if not target:
        print('[jac-ssh] no target URL provided or saved.', file=sys.stderr)
        return 2
    ws_url = gateway_ws_url(target, token=token or sess.get('token'))
    hdrs = dict(sess.get('headers') or {})
    if headers: hdrs.update(headers)
    try:
        tok = bearer or sess.get('bearer') or (_read_token_file(tf) if (tf := token_file or sess.get('token_file')) else None)
        if tok: hdrs['Authorization'] = f'Bearer {tok}'
        hdrs[pubkey_header or sess.get('pubkey_header') or COLAB_PUBKEY_HEADER] = resolve_pubkey(identity or sess.get('identity'))
    except Exception as e:
        print(f'[jac-ssh] {e}', file=sys.stderr)
        return 2
    try:
        with ws_connect(ws_url, additional_headers=hdrs, compression=None, max_size=None, ping_interval=20) as ws:
            return bridge_ws(ws, stdin_fd=stdin_fd, stdout=stdout)
    except InvalidStatus as e:
        print(f'[jac-ssh] {_explain_handshake_error(e.response.status_code, e.response.body or b"")}', file=sys.stderr)
        return 1
    except (WebSocketException, OSError) as e:
        print(f'[jac-ssh] WebSocket connection failed: {e}', file=sys.stderr)
        return 1

Start a local WebSocket echo server that records upgrade headers and rejects `/deny` with HTTP 401.

In [ ]:
seen = {}
def _echo_handler(ws):
    for msg in ws: ws.send(msg)

def _check_req(conn, req):
    seen['headers'] = dict(req.headers)
    if req.path.startswith('/deny'): return WsResponse(401, 'Unauthorized', WsHeaders(), b'bad token')

srv = ws_serve(_echo_handler, '127.0.0.1', 0, process_request=_check_req)
ws_port = srv.socket.getsockname()[1]
threading.Thread(target=srv.serve_forever, daemon=True).start()

Run `proxy_ssh` over a pair of pipes and verify the binary round-trip, upgrade headers, and denial exit code.

In [ ]:
td_ws = tempfile.TemporaryDirectory(prefix='jac-ssh-ws-')
kp, tf = Path(td_ws.name)/'id_ed25519', Path(td_ws.name)/'session'
subprocess.run(['ssh-keygen', '-q', '-t', 'ed25519', '-N', '', '-f', str(kp)], check=True)
tf.write_text('secret-cookie-value\n')
r_in, w_in = os.pipe()
r_out, w_out = os.pipe()
out_buf, got = os.fdopen(w_out, 'wb'), []
def _drive():
    os.write(w_in, b'SSH-2.0-test\r\n')
    got.append(os.read(r_out, 1024))
    os.close(w_in)
t = threading.Thread(target=_drive, daemon=True)
t.start()
rc = proxy_ssh(url=f'ws://127.0.0.1:{ws_port}/api/ssh', identity=kp, token_file=tf, stdin_fd=r_in, stdout=out_buf)
t.join(5)
out_buf.close()
os.close(r_in)
os.close(r_out)
test_eq((rc, got[0]), (0, b'SSH-2.0-test\r\n'))
test_eq(seen['headers'].get('x-colab-ssh-pubkey'), kp.with_suffix('.pub').read_text().strip())
test_eq(seen['headers'].get('authorization'), 'Bearer secret-cookie-value')
test_eq(proxy_ssh(url=f'ws://127.0.0.1:{ws_port}/deny/ssh', identity=kp, token_file=tf), 1)

In [ ]:
#| hide
srv.shutdown()
td_ws.cleanup()

## `jac-ssh` CLI

`jac-ssh` updates `~/.ssh/config` for `--colab` (`User root`, `/colab/ssh`), `--solveit` (`User solveit`, `/rusty/api/ssh`, `~/.config/solveit/session`), or a bare Rustygate URL, and runs as OpenSSH's `ProxyCommand` when called with `--proxy`.

In [ ]:
#| export
SOLVEIT_SESSION_FILE = xdg_config_home()/'solveit'/'session'

@call_parse
def main(
    host:str=None, # SSH Host alias (defaults to 'colab' with --colab, 'solveit' with --solveit, else 'rustygate')
    url:str=None, # Target URL (Colab runtime URL, Solveit instance URL, or Rustygate URL)
    colab:bool=False, # Configure for a Google Colab runtime
    solveit:bool=False, # Configure for a Solveit instance (uses ~/.config/solveit/session and /rusty/api/ssh)
    proxy:bool=False, # Run as an OpenSSH ProxyCommand bridging stdin/stdout to the WebSocket
    user:str=None, # Remote SSH username (defaults to 'root' for --colab, 'solveit' for --solveit)
    identity:str=None, # Path to ed25519/ecdsa key (auto-detected from ~/.ssh if omitted)
    token:str=None, # Query token (Colab runtime proxy token or Rustygate gateway token)
    bearer:str=None, # Explicit Authorization Bearer token
    token_file:str=None, # File containing a Bearer token (defaults to ~/.config/solveit/session with --solveit)
    ssh_config:str='~/.ssh/config', # OpenSSH config file to update
    store:str=None, # Session store JSON path (defaults to $XDG_CONFIG_HOME/jupyasyncclient/ssh.json)
    replace:bool=False, # Replace an existing Host block in ssh_config
):
    "Configure `~/.ssh/config` or bridge OpenSSH `ProxyCommand` over WebSockets for Colab, Solveit, and Rustygate."
    if colab and solveit: raise SystemExit('Pass at most one of --colab or --solveit.')
    if proxy:
        ws_url = colab_ws_url(url, token) if (colab and url and token) else (solveit_ws_url(url) if (solveit and url) else url)
        hdrs = {'X-Colab-Client-Agent': 'colab-cli'} if colab else None
        tf = token_file or (SOLVEIT_SESSION_FILE if solveit and not bearer and not host else None)
        raise SystemExit(proxy_ssh(host, url=ws_url, identity=identity, token=None if colab else token,
            bearer=bearer, token_file=tf, headers=hdrs, store=store))
    if colab:
        h = host or 'colab'
        ws_url = colab_ws_url(url, token) if (url and token) else url
        hdrs = {'X-Colab-Client-Agent': 'colab-cli'} if ws_url else None
        cmd = setup_ssh(h, ws_url, user=user or 'root', identity=identity, headers=hdrs,
            ssh_config=ssh_config, store=store, replace=replace)
    elif solveit:
        h = host or 'solveit'
        ws_url = solveit_ws_url(url) if url else None
        tf = token_file or (None if bearer else SOLVEIT_SESSION_FILE)
        cmd = setup_ssh(h, ws_url, user=user or 'solveit', identity=identity, bearer=bearer,
            token_file=tf if (ws_url or token_file) else None, ssh_config=ssh_config, store=store, replace=replace)
    else:
        h = host or 'rustygate'
        ws_url = gateway_ws_url(url, token=token) if url else None
        cmd = setup_ssh(h, ws_url, user=user, identity=identity, bearer=bearer, token_file=token_file,
            ssh_config=ssh_config, store=store, replace=replace)
    print(cmd)

Verify the CLI flags for `--colab` and `--solveit` against temporary config and session files.

In [ ]:
with tempfile.TemporaryDirectory(prefix='jac-ssh-cli-') as td:
    cfg, st = Path(td)/'config', Path(td)/'ssh.json'
    base = [sys.executable, '-m', 'jupyasyncclient.ssh', '--ssh-config', str(cfg), '--store', str(st)]
    r1 = subprocess.run(base + ['--colab'], check=True, capture_output=True, text=True)
    r2 = subprocess.run(base + ['--solveit', '--host', 'myinst', '--url', 'https://myinst.solve.it.com'], check=True, capture_output=True, text=True)
    test_eq((r1.stdout.strip(), r2.stdout.strip()), ('ssh colab', 'ssh myinst'))
    cfg_txt = cfg.read_text()
    assert 'Host colab' in cfg_txt and 'User root' in cfg_txt
    assert 'Host myinst' in cfg_txt and 'User solveit' in cfg_txt
    test_eq(get_ssh_session('myinst', store=st).url, 'wss://myinst.solve.it.com/rusty/api/ssh')

In [ ]:
#| hide
import nbdev
nbdev.nbdev_export()